<a href="https://colab.research.google.com/github/Kathyayini26/flyrank-ml/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Kathyayini26/flyrank-ml/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of Analysis:** One row represents one daily performance record for a specific piece of content, identified by its report date, client and content ID.

**Time Window:** I will use March 2026 (`2026-03`) as my development month to analyse content performance and identify webpages that may need review.

**Reason:** Using a consistent monthly window helps me examine content performance patterns while avoiding the final June 2026 sample, which is reserved as a sealed test month.

In [ ]:

# ============================================================
# WEEK 3 - SECTION 1: UNIT OF ANALYSIS + TIME WINDOW
# Project Lane: Content Opportunity Scoring
# Dataset: FlyRank/internship-warehouse
# Table: fact_content_daily_performance
# Month: March 2026
# ============================================================

# 1. Install required libraries
%pip -q install -U duckdb huggingface_hub

import duckdb
import pandas as pd
import getpass

# 2. Authenticate directly with Hugging Face
HF_TOKEN = getpass.getpass("Enter your Hugging Face READ token: ")

if not HF_TOKEN:
    raise ValueError("Hugging Face token is required.")

# 3. Connect to DuckDB
con = duckdb.connect()

# Configure Hugging Face authentication
safe_token = HF_TOKEN.replace("'", "''")

con.execute(f"""
    CREATE OR REPLACE SECRET hf_secret (
        TYPE huggingface,
        TOKEN '{safe_token}'
    )
""")

print("Hugging Face authentication configured successfully.")

# 4. Select March 2026 warehouse partition
dataset_path = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/**/*.parquet"
)

# 5. Inspect actual warehouse schema
schema_query = f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{dataset_path}',
        hive_partitioning = true
    )
"""

schema = con.execute(schema_query).df()

print("\nWAREHOUSE TABLE SCHEMA")
print("-" * 60)
display(schema)

# 6. Identify actual grain columns
columns = schema["column_name"].tolist()
lower_columns = {col.lower(): col for col in columns}

date_candidates = ["report_date", "date", "event_date"]

date_col = next(
    (lower_columns[c] for c in date_candidates if c in lower_columns),
    None
)

client_col = next(
    (col for col in columns
     if "client" in col.lower()
     and any(x in col.lower() for x in ["hash", "id", "key"])),
    None
)

content_col = next(
    (col for col in columns
     if "content" in col.lower()
     and any(x in col.lower() for x in ["hash", "id", "key"])),
    None
)

print("\nIDENTIFIED GRAIN COLUMNS")
print("-" * 60)
print("Date column:", date_col)
print("Client column:", client_col)
print("Content column:", content_col)

if not all([date_col, client_col, content_col]):
    raise ValueError(
        "Could not identify all grain columns. "
        "Check the actual schema displayed above."
    )

# 7. Verify unit of analysis
grain_query = f"""
    WITH grouped AS (
        SELECT
            "{date_col}",
            "{client_col}",
            "{content_col}",
            COUNT(*) AS rows_per_key
        FROM read_parquet(
            '{dataset_path}',
            hive_partitioning = true
        )
        GROUP BY
            "{date_col}",
            "{client_col}",
            "{content_col}"
    )

    SELECT
        COUNT(*) AS unique_grain_keys,
        SUM(rows_per_key) AS total_rows,
        SUM(
            CASE
                WHEN rows_per_key > 1
                THEN rows_per_key - 1
                ELSE 0
            END
        ) AS duplicate_rows,
        MAX(rows_per_key) AS maximum_rows_per_key
    FROM grouped
"""

grain_result = con.execute(grain_query).df()

print("\nUNIT OF ANALYSIS VERIFICATION")
print("-" * 60)
display(grain_result)

# 8. Verify March 2026 date range
date_query = f"""
    SELECT
        MIN("{date_col}") AS first_date,
        MAX("{date_col}") AS last_date,
        COUNT(DISTINCT "{date_col}") AS number_of_dates
    FROM read_parquet(
        '{dataset_path}',
        hive_partitioning = true
    )
"""

date_result = con.execute(date_query).df()

print("\nSELECTED TIME WINDOW")
print("-" * 60)
display(date_result)

# 9. Display data contract
print("\nDATA CONTRACT")
print("-" * 60)
print("Lane: Content Opportunity Scoring")
print("Table: fact_content_daily_performance")
print("Time window: March 2026")
print(
    "Unit: One row per report date, "
    "client and content item."
)

print("\nSECTION 1 COMPLETED")


Enter your Hugging Face READ token: ··········
Hugging Face authentication configured successfully.

WAREHOUSE TABLE SCHEMA
------------------------------------------------------------


,column_name,column_type,null,key,default,extra
0,report_date,DATE,YES,None,None,None
1,client_hash_id,VARCHAR,YES,None,None,None
2,content_hash_id,VARCHAR,YES,None,None,None
3,client_has_gsc,BOOLEAN,YES,None,None,None
4,client_has_ga4,BOOLEAN,YES,None,None,None
5,gsc_data_available,BOOLEAN,YES,None,None,None
6,ga4_data_available,BOOLEAN,YES,None,None,None
7,gsc_impressions,BIGINT,YES,None,None,None
8,gsc_clicks,BIGINT,YES,None,None,None
9,gsc_sum_position,BIGINT,YES,None,None,None



IDENTIFIED GRAIN COLUMNS
------------------------------------------------------------
Date column: report_date
Client column: client_hash_id
Content column: content_hash_id


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


UNIT OF ANALYSIS VERIFICATION
------------------------------------------------------------


,unique_grain_keys,total_rows,duplicate_rows,maximum_rows_per_key
0,9841378,9841378.0,0.0,1



SELECTED TIME WINDOW
------------------------------------------------------------


,first_date,last_date,number_of_dates
0,2026-03-01,2026-03-31,31



DATA CONTRACT
------------------------------------------------------------
Lane: Content Opportunity Scoring
Table: fact_content_daily_performance
Time window: March 2026
Unit: One row per report date, client and content item.

SECTION 1 COMPLETED


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

Field Classification
- Features: Historical performance metrics available before the content review decision, such as impressions, clicks, sessions and engagement.
- Label: A future performance outcome or a proxy indicating whether content performance declines.
- Context: Content ID, client ID and report date, used to identify and organise records.
- Excluded: Future outcome information, identifiers that do not provide predictive value, and fields that could cause data leakage.

In [ ]:

# ============================================================
# WEEK 3 - SECTION 2: FIELD CLASSIFICATION
# Feature / Label / Context / Excluded
# Dataset: Actual FlyRank Hugging Face Warehouse
# ============================================================

import pandas as pd

# 1. Read the actual table schema
schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{dataset_path}',
        hive_partitioning = true
    )
""").df()

available_columns = schema["column_name"].tolist()

print("TOTAL AVAILABLE COLUMNS:", len(available_columns))

# 2. Define feature candidates
feature_candidates = [
    "impressions",
    "clicks",
    "sessions",
    "users",
    "pageviews",
    "engaged_sessions",
    "engagement_rate",
    "ctr",
    "avg_position",
    "content_age_days",
    "days_since_last_update"
]

# 3. Define label candidates
label_candidates = [
    "trend_direction",
    "trend_pct",
    "future_clicks",
    "future_impressions",
    "future_sessions"
]

# 4. Define context candidates
context_candidates = [
    "report_date",
    "client_hash_id",
    "content_hash_id",
    "month"
]

# 5. Define excluded candidates
excluded_candidates = [
    "future_performance",
    "target",
    "label",
    "outcome"
]

# 6. Match candidates with actual available columns
def match_columns(candidates):
    return [
        col for col in available_columns
        if col.lower() in [x.lower() for x in candidates]
    ]

features = match_columns(feature_candidates)
labels = match_columns(label_candidates)
context = match_columns(context_candidates)
excluded = match_columns(excluded_candidates)

# 7. Display the field classification
field_classification = []

for col in features:
    field_classification.append({
        "Field": col,
        "Category": "Feature",
        "Reason": "Historical performance information"
    })

for col in labels:
    field_classification.append({
        "Field": col,
        "Category": "Label candidate",
        "Reason": "Potential performance outcome; verify timing"
    })

for col in context:
    field_classification.append({
        "Field": col,
        "Category": "Context",
        "Reason": "Identifies the content, client or reporting period"
    })

for col in excluded:
    field_classification.append({
        "Field": col,
        "Category": "Excluded",
        "Reason": "Potential target leakage or outcome information"
    })

classification_df = pd.DataFrame(field_classification)

print("\nFIELD CLASSIFICATION")
print("-" * 60)
display(classification_df)

# 8. Display unclassified columns for review
classified_columns = (
    features + labels + context + excluded
)

unclassified = [
    col for col in available_columns
    if col not in classified_columns
]

print("\nOTHER AVAILABLE COLUMNS TO REVIEW:")
print(unclassified)

# 9. Summary
print("\nCLASSIFICATION SUMMARY")
print("-" * 60)
print("Features:", len(features))
print("Label candidates:", len(labels))
print("Context fields:", len(context))
print("Excluded candidates:", len(excluded))
print("Fields requiring review:", len(unclassified))


TOTAL AVAILABLE COLUMNS: 31

FIELD CLASSIFICATION
------------------------------------------------------------


,Field,Category,Reason
0,report_date,Context,"Identifies the content, client or reporting pe..."
1,client_hash_id,Context,"Identifies the content, client or reporting pe..."
2,content_hash_id,Context,"Identifies the content, client or reporting pe..."
3,month,Context,"Identifies the content, client or reporting pe..."



OTHER AVAILABLE COLUMNS TO REVIEW:
['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

CLASSIFICATION SUMMARY
------------------------------------------------------------
Features: 0
Label candidates: 0
Context fields: 4
Excluded candidates: 0
Fields requiring review: 27


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

We need to verify four things using the actual March 2026 Hugging Face warehouse data:
1. Grain – one row per date, client and content.
2. Row count – how many records are available.
3. Missing values – how complete are the selected fields?
4. Time window – earliest and latest reporting dates.

In [ ]:

# ============================================================
# WEEK 3 - SECTION 3: VERIFY DATA CONTRACT WITH QUERIES
# Dataset: FlyRank Hugging Face Warehouse
# Table: fact_content_daily_performance
# Month: March 2026
# ============================================================

# 1. Verify the actual table schema
schema = con.execute(f"""
    DESCRIBE
    SELECT *
    FROM read_parquet(
        '{dataset_path}',
        hive_partitioning = true
    )
""").df()

columns = schema["column_name"].tolist()

print("AVAILABLE DATA COLUMNS")
print("-" * 60)
print(columns)

# Identify actual grain columns
lower_columns = {col.lower(): col for col in columns}

date_col = next(
    (lower_columns[c] for c in ["report_date", "date", "event_date"]
     if c in lower_columns),
    None
)

client_col = next(
    (col for col in columns
     if "client" in col.lower()
     and any(x in col.lower() for x in ["hash", "id", "key"])),
    None
)

content_col = next(
    (col for col in columns
     if "content" in col.lower()
     and any(x in col.lower() for x in ["hash", "id", "key"])),
    None
)

if not all([date_col, client_col, content_col]):
    raise ValueError("Could not identify the actual grain columns.")

# 2. QUERY 1: Verify the grain
grain_query = f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT (
            "{date_col}",
            "{client_col}",
            "{content_col}"
        )) AS unique_grain_keys,
        COUNT(*) -
        COUNT(DISTINCT (
            "{date_col}",
            "{client_col}",
            "{content_col}"
        )) AS duplicate_rows
    FROM read_parquet(
        '{dataset_path}',
        hive_partitioning = true
    )
"""

grain_result = con.execute(grain_query).df()

print("\nQUERY 1: GRAIN VERIFICATION")
print("-" * 60)
display(grain_result)

# 3. QUERY 2: Verify row count and date span
count_query = f"""
    SELECT
        COUNT(*) AS total_rows,
        MIN("{date_col}") AS first_date,
        MAX("{date_col}") AS last_date,
        COUNT(DISTINCT "{date_col}") AS distinct_dates
    FROM read_parquet(
        '{dataset_path}',
        hive_partitioning = true
    )
"""

count_result = con.execute(count_query).df()

print("\nQUERY 2: ROW COUNT AND DATE SPAN")
print("-" * 60)
display(count_result)

# 4. QUERY 3: Verify missing values
# Select up to five actual available performance columns
preferred_fields = [
    "impressions",
    "clicks",
    "sessions",
    "pageviews",
    "engagement_rate"
]

selected_fields = [
    lower_columns[field]
    for field in preferred_fields
    if field in lower_columns
]

if not selected_fields:
    selected_fields = [
        col for col in columns
        if col.lower() not in [
            date_col.lower(),
            client_col.lower(),
            content_col.lower()
        ]
    ][:5]

missing_expressions = ", ".join(
    [
        f"COUNT(*) FILTER (WHERE \"{col}\" IS NULL) "
        f"AS \"{col}_missing\""
        for col in selected_fields
    ]
)

missing_query = f"""
    SELECT
        COUNT(*) AS total_rows,
        {missing_expressions}
    FROM read_parquet(
        '{dataset_path}',
        hive_partitioning = true
    )
"""

missing_result = con.execute(missing_query).df()

print("\nQUERY 3: MISSING VALUE VERIFICATION")
print("-" * 60)
display(missing_result)

# 5. Final verification summary
print("\nDATA CONTRACT VERIFICATION SUMMARY")
print("-" * 60)
print("Dataset: FlyRank/internship-warehouse")
print("Table: fact_content_daily_performance")
print("Partition: March 2026")
print("Date column:", date_col)
print("Client column:", client_col)
print("Content column:", content_col)
print("Fields checked for missing values:", selected_fields)

print("\nAll three verification queries executed.")


AVAILABLE DATA COLUMNS
------------------------------------------------------------
['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events', 'month']


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


QUERY 1: GRAIN VERIFICATION
------------------------------------------------------------


,total_rows,unique_grain_keys,duplicate_rows
0,9841378,9841378,0



QUERY 2: ROW COUNT AND DATE SPAN
------------------------------------------------------------


,total_rows,first_date,last_date,distinct_dates
0,9841378,2026-03-01,2026-03-31,31


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


QUERY 3: MISSING VALUE VERIFICATION
------------------------------------------------------------


,total_rows,client_has_gsc_missing,client_has_ga4_missing,gsc_data_available_missing,ga4_data_available_missing,gsc_impressions_missing
0,9841378,0,0,0,3018741,0



DATA CONTRACT VERIFICATION SUMMARY
------------------------------------------------------------
Dataset: FlyRank/internship-warehouse
Table: fact_content_daily_performance
Partition: March 2026
Date column: report_date
Client column: client_hash_id
Content column: content_hash_id
Fields checked for missing values: ['client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions']

All three verification queries executed.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

**Data Limitations**

1. **Unbalanced history:** Not all clients have the same amount of historical data, so comparisons may not be equally reliable.
2. **GSC-only early records:** Some records may have Google Search Console data without corresponding Google Analytics 4 data.
3. **Missing GA4 availability:** The March dataset contains 3,018,741 missing values in `ga4_data_available`. These must not automatically be treated as unavailable or false.
4. **Window overlap:** If historical features and future outcome labels use overlapping dates, the model may learn information from the outcome period, causing data leakage.
5. **No causal proof:** The data can reveal performance patterns, but cannot prove that refreshing content will cause traffic or engagement to improve.

These limitations must be considered when selecting features, defining labels and evaluating the model.

In [ ]:

# ============================================================
# WEEK 3 - SECTION 4: DATA LIMITATIONS
# Actual FlyRank Hugging Face Warehouse
# ============================================================

# 1. Check GSC and GA4 availability combinations

availability_query = f"""
SELECT
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available,
    COUNT(*) AS total_rows
FROM read_parquet(
    '{dataset_path}',
    hive_partitioning = true
)
GROUP BY
    client_has_gsc,
    client_has_ga4,
    gsc_data_available,
    ga4_data_available
ORDER BY total_rows DESC
"""

availability_df = con.execute(availability_query).df()

print("GSC AND GA4 AVAILABILITY")
print("-" * 60)
display(availability_df)

# 2. Check missing values in important fields

missing_query = f"""
SELECT
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS NULL
    ) AS missing_ga4_availability,

    COUNT(*) FILTER (
        WHERE ga4_sessions IS NULL
    ) AS missing_ga4_sessions,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS NULL
    ) AS missing_gsc_availability,

    COUNT(*) FILTER (
        WHERE gsc_impressions IS NULL
    ) AS missing_gsc_impressions

FROM read_parquet(
    '{dataset_path}',
    hive_partitioning = true
)
"""

missing_df = con.execute(missing_query).df()

print("\nMISSING VALUE ANALYSIS")
print("-" * 60)
display(missing_df)

# 3. Compare daily GSC and GA4 availability

daily_query = f"""
SELECT
    report_date,
    COUNT(*) AS total_rows,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS gsc_available_rows,

    COUNT(*) FILTER (
        WHERE ga4_data_available IS TRUE
    ) AS ga4_available_rows

FROM read_parquet(
    '{dataset_path}',
    hive_partitioning = true
)
GROUP BY report_date
ORDER BY report_date
"""

daily_df = con.execute(daily_query).df()

print("\nDAILY DATA AVAILABILITY")
print("-" * 60)
display(daily_df)

# 4. Summarise the limitations

print("\nDATA LIMITATION SUMMARY")
print("-" * 60)

print("1. Client data availability may differ.")
print("2. GSC and GA4 coverage may not be identical.")
print("3. Missing GA4 values require careful handling.")
print("4. Overlapping feature and outcome windows can cause leakage.")
print("5. Historical performance does not establish causation.")

print("\nSECTION 4 COMPLETED")




GSC AND GA4 AVAILABILITY
------------------------------------------------------------


,client_has_gsc,client_has_ga4,gsc_data_available,ga4_data_available,total_rows
0,True,True,False,False,4690323
1,True,True,True,False,1718348
2,True,False,True,<NA>,1528366
3,True,False,False,<NA>,1490375
4,True,True,True,True,364347
5,True,True,False,True,49619


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))


MISSING VALUE ANALYSIS
------------------------------------------------------------


,total_rows,missing_ga4_availability,missing_ga4_sessions,missing_gsc_availability,missing_gsc_impressions
0,9841378,3018741,3018741,0,0



DAILY DATA AVAILABILITY
------------------------------------------------------------


,report_date,total_rows,gsc_available_rows,ga4_available_rows
0,2026-03-01,275874,101910,4909
1,2026-03-02,276269,103696,5349
2,2026-03-03,311676,107362,6831
3,2026-03-04,311675,109377,6791
4,2026-03-05,311676,109740,7653
5,2026-03-06,312187,110037,8389
6,2026-03-07,312387,102153,7865
7,2026-03-08,313374,101170,7589
8,2026-03-09,313874,111313,9653
9,2026-03-10,314047,113051,11731



DATA LIMITATION SUMMARY
------------------------------------------------------------
1. Client data availability may differ.
2. GSC and GA4 coverage may not be identical.
3. Missing GA4 values require careful handling.
4. Overlapping feature and outcome windows can cause leakage.
5. Historical performance does not establish causation.

SECTION 4 COMPLETED


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.